# Analisis exploratorio de ventas (puntos 2.a y 2.b)

Este notebook obtiene los datos desde PostgreSQL y calcula media, mediana y moda para las variables cuantitativas. Los resultados de Pandas se contrastan de forma independiente con SQL.

## Criterio de seleccion

Se analizan `edad`, `venta_total`, `n_compras`, `monto_compra` y `tiempo`. Se excluyen identificadores y categorias codificadas (`genero`, `metodo_pago`, `navegador`, `boletin` y `vale`) porque sus promedios no tienen una interpretacion valida.

In [1]:
import sys
from pathlib import Path
from IPython.display import display

RAIZ_PRACTICA = Path.cwd().resolve()
if not (RAIZ_PRACTICA / 'app').is_dir():
    RAIZ_PRACTICA = RAIZ_PRACTICA.parent
if not (RAIZ_PRACTICA / 'app').is_dir():
    raise FileNotFoundError('Ejecute el notebook desde Practica_1 o su carpeta del punto 2.')
sys.path.insert(0, str(RAIZ_PRACTICA))

from app.analisis.punto_02 import ejecutar_analisis

resultado = ejecutar_analisis(
    directorio_resultados=RAIZ_PRACTICA / '02-Analisis-exploratorio' / 'resultados'
)

## 2.a Obtencion desde la base de datos

La funcion anterior abre una transaccion de solo lectura y consulta `public.clientes` y `public.compras` por separado.

In [2]:
print(f"Clientes obtenidos: {len(resultado['clientes']):,}")
print(f"Compras obtenidas:  {len(resultado['compras']):,}")
print(f"Transaccion de solo lectura: {resultado['control']['solo_lectura']}")
display(resultado['clientes'].head())
display(resultado['compras'].head())

Clientes obtenidos: 6,500
Compras obtenidas:  6,500
Transaccion de solo lectura: on


,id_cliente,edad,genero,venta_total,n_compras
0,504308,18,1,737.4,8
1,504309,46,1,689.2,17
2,504310,38,0,95.6,1
3,504311,44,1,75.7,2
4,504312,41,0,46.0,2


,id_compra,id_cliente,fecha_compra,monto_compra,metodo_pago,tiempo,navegador,boletin,vale
0,1,504308,2021-02-02,109.054,2,1022,0,True,False
1,2,504309,2021-10-10,35.326,1,865,0,False,False
2,3,504310,2021-08-27,89.353,2,900,2,True,True
3,4,504311,2021-06-10,27.220,1,793,0,True,False
4,5,504312,2021-06-08,28.531,0,729,0,False,False


In [3]:
print('Tipos de clientes:')
display(resultado['clientes'].dtypes.rename('tipo').to_frame())
print('Tipos de compras:')
display(resultado['compras'].dtypes.rename('tipo').to_frame())
print('Validaciones de extraccion e integridad:')
display(resultado['validaciones'])

Tipos de clientes:


,tipo
id_cliente,int64
edad,int64
genero,int64
venta_total,float64
n_compras,int64


Tipos de compras:


,tipo
id_compra,int64
id_cliente,int64
fecha_compra,datetime64[s]
monto_compra,float64
metodo_pago,int64
tiempo,int64
navegador,int64
boletin,boolean
vale,boolean


Validaciones de extraccion e integridad:


,validacion,cumple,detalle
0,clientes no vacio,True,6500 filas
1,compras no vacio,True,6500 filas
2,id_cliente unico,True,6500 identificadores unicos
3,id_compra unico,True,6500 identificadores unicos
4,sin nulos en clientes,True,0 valores nulos
5,sin nulos en compras,True,0 valores nulos
6,sin compras huerfanas,True,0 huerfanas
7,dominio de genero,True,"valores: [0, 1]"
8,dominio de metodo_pago,True,"valores: [0, 1, 2]"
9,dominio de navegador,True,"valores: [0, 1, 2, 3, 4]"


## 2.b Estadisticas basicas

No se redondean los datos antes del calculo. La columna `moda` conserva todos los empates y `frecuencia_moda` indica cuantas veces aparece cada valor modal.

In [4]:
columnas = ['tabla', 'variable', 'unidad', 'cantidad', 'nulos', 'media', 'mediana', 'moda', 'frecuencia_moda', 'cantidad_modas']
display(resultado['estadisticas'][columnas].style.format({'media': '{:.4f}', 'mediana': '{:.4f}'}))

,tabla,variable,unidad,cantidad,nulos,media,mediana,moda,frecuencia_moda,cantidad_modas
0,clientes,edad,anios,6500,0,36.3052,36.0000,18,465,1
1,clientes,venta_total,moneda,6500,0,206.2424,137.3500,98,12,1
2,clientes,n_compras,compras,6500,0,5.0900,4.0000,2,1044,1
3,compras,monto_compra,moneda,6500,0,39.7871,35.7640,37.145,5,1
4,compras,tiempo,segundos,6500,0,767.3762,768.0000,852,24,1


## Verificacion independiente con PostgreSQL

PostgreSQL recalcula las medidas con `AVG`, `percentile_cont(0.5)` y frecuencias agrupadas. Todas las columnas `todo_coincide` deben ser verdaderas.

In [5]:
columnas_contraste = ['tabla', 'variable', 'media_coincide', 'mediana_coincide', 'moda_coincide', 'frecuencia_moda_coincide', 'todo_coincide']
display(resultado['contraste'][columnas_contraste])
assert resultado['validaciones']['cumple'].all()
assert resultado['contraste']['todo_coincide'].all()
print('Analisis verificado: la extraccion es integra y Pandas coincide con PostgreSQL.')

,tabla,variable,media_coincide,mediana_coincide,moda_coincide,frecuencia_moda_coincide,todo_coincide
0,clientes,edad,True,True,True,True,True
1,clientes,venta_total,True,True,True,True,True
2,clientes,n_compras,True,True,True,True,True
3,compras,monto_compra,True,True,True,True,True
4,compras,tiempo,True,True,True,True,True


Analisis verificado: la extraccion es integra y Pandas coincide con PostgreSQL.


## Nota de interpretacion

`venta_total` y `n_compras` son acumulados anuales del cliente. `monto_compra` y `fecha_compra` describen la fila registrada en `compras`. Las tablas se mantienen separadas para no duplicar los acumulados de cliente si se agregan mas compras en el futuro.